In [2]:
from langchain_community.utilities import SQLDatabase
from langchain_community.tools import QuerySQLDataBaseTool
from langchain.chains import create_sql_query_chain
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
import getpass
import os

In [3]:
db = SQLDatabase.from_uri("sqlite:///D:/AI_team/github/Vision_AI_RnD_team/projects/pj6(createDB)/DB/sqlLite/scenario_1128.db")

In [4]:
print(db.get_usable_table_names())

['civil_elements', 'friendly', 'main_event', 'mission', 'oppose', 'weather']


In [6]:
db.run("select * from friendly;")

"[('tick', 'unit_id', 'unit_name', 'echelon', 'position_lat', 'position_lon', 'personnel', 'equipments', 'personnel_strength', 'equipment_status', 'ammo_supply', 'fuel_supply', 'morale', 'current_task'), ('5', 'kpa_001', 'KPA_LEADER', '사단', '355.2', '110.1', '7000', '소총', '7000', '100', '100', '100', '100', 'prepare'), ('5', 'kpa_002', 'KPA_SECURITY', '대대', '420', '165.8', '300', '소총', '300', '100', '100', '85', '100', 'prepare'), ('5', 'kpa_003', 'KPA_SECURITY2', '여단', '162.2', '101.4', '1500', '소총', '1500', '100', '100', '80', '100', 'prepare'), ('5', 'kpa_004', 'KPA_SUPPORT', '중대', '235.8', '192.9', '60', '소총', '60', '100', '100', '95', '100', 'move'), ('5', 'kpa_005', 'KPA_SUPPORT2', '대대', '263.3', '212.3', '300', '소총', '300', '100', '100', '88', '100', 'move'), ('5', 'kpa_006', 'KPA_ARTILLERY', '여단', '138.9', '239.7', '1500', '박격포', '1500', '100', '100', '100', '100', 'prepare'), ('5', 'kpa_007', 'KPA_BACKUP', '여단', '46', '223.5', '1500', '소총', '1500', '100', '100', '100', '100', 

In [7]:
llm  = ChatOpenAI(
            api_key="ai",
            model="openai/gpt-oss-20b",
            base_url="http://192.168.0.110:8000/v1",
            temperature=0
        )

In [15]:
sql_query_gen_chain = create_sql_query_chain(llm, db)

response = sql_query_gen_chain.invoke({"question": "시간이 0에서 1까지의 데이터를 출력해줘"})

In [16]:
print(response)

Question: 시간이 0에서 1까지의 데이터를 출력해줘  
SQLQuery: SELECT "field1","field2","field3" FROM tab1 WHERE "field1" BETWEEN 0 AND 1 LIMIT 5;  
SQLResult:


In [12]:
clean_sql_prompt_template = """You are an expert in SQL Lite. You are asked to fix badly formedSQL Lite queries,
which might contain unneeded prefixes or suffixes. Given the following unclean SQL statement, transform it to a clean, 
excutable SQL statement for SQL lite. 
Only return an executable SQL statement which terminates with a semicolon. 
Do not return anything else. Do not include the language name or symbol like ```.

Unclean SQL: {unclean_sql}"""

clean_sql_prompt = ChatPromptTemplate.from_template(clean_sql_prompt_template)

clean_sql_chain = clean_sql_prompt | llm

full_sql_gen_chain = sql_query_gen_chain | clean_sql_chain | StrOutputParser()

In [18]:
question = "시간이 0에서 1까지의 모든 항목을 출력해줘"

response = full_sql_gen_chain.invoke({"question": question})

print(response)

SELECT field1, field2, field3, field4, field5, field6, field7, field8, field9 FROM tab1 WHERE field1 BETWEEN 0 AND 1;


In [23]:
question = "시간이 4에서 5까지의 모든 항목을 출력해줘"

sql_query_exec_chain = QuerySQLDataBaseTool(db=db)

sql_query_gen_and_exec_chain = full_sql_gen_chain | sql_query_exec_chain | StrOutputParser()

response = sql_query_gen_and_exec_chain.invoke({"question": question})

In [24]:
print(response)

[('4', '0', 'KPA_PLATOON1', '46', '223.5', '0', '0', '100', 'CDET 다른 1개 보안부대가 서쪽 측면에서 적 보안부대 전진 관찰 후 진지 점령, 박격포 부대에 고정임무 하달'), ('4', '0', 'KPA_PLATOON2', '162.2', '101.4', '130', '0', '100', 'CDET 다른 1개 보안부대가 서쪽 측면에서 적 보안부대 전진 관찰 후 진지 점령, 박격포 부대에 고정임무 하달'), ('4', '0', 'KPA_LEADER', '95.2', '194.9', '0', '0', '100', 'CDET 다른 1개 보안부대가 서쪽 측면에서 적 보안부대 전진 관찰 후 진지 점령, 박격포 부대에 고정임무 하달'), ('4', '0', 'KPA_ARTILLERY', '138.9', '239.7', '0', '0', '100', 'CDET 다른 1개 보안부대가 서쪽 측면에서 적 보안부대 전진 관찰 후 진지 점령, 박격포 부대에 고정임무 하달'), ('4', '0', 'KPA_SQUAD', '197.8', '181.9', '0', '0', '100', 'CDET 다른 1개 보안부대가 서쪽 측면에서 적 보안부대 전진 관찰 후 진지 점령, 박격포 부대에 고정임무 하달'), ('4', '0', 'KPA_SECTION', '242.3', '230.3', '0', '0', '100', 'CDET 다른 1개 보안부대가 서쪽 측면에서 적 보안부대 전진 관찰 후 진지 점령, 박격포 부대에 고정임무 하달'), ('4', '0', 'KPA_SECTION2', '420', '165.8', '130', '0', '100', 'CDET 다른 1개 보안부대가 서쪽 측면에서 적 보안부대 전진 관찰 후 진지 점령, 박격포 부대에 고정임무 하달'), ('4', '1', 'ROK_SQUAD', '310.4', '165.6', '0', '2', '100', 'CDET 다른 1개 보안부대가 서쪽 측면에서 적 보안부대 전진 관찰 후 진지 

In [28]:
scenario_explain_template = """
### 1. 프롬프트 제목
**[군사 시뮬레이션 데이터 기반] 시나리오 상황 분석 및 묘사 전문가 프롬프트**

### 2. 역할 및 목표 설정
"당신은 **최소 20년 경력의 군사 전술 분석 전문가**이자 **시뮬레이션 데이터 해석관**입니다. 당신의 임무는 제공된 테이블 형태의 시뮬레이션 데이터를 **단순한 수치 나열이 아닌**, 시간 흐름에 따른 **생생하고 전술적인 교전 상황 묘사**로 전환하는 것입니다. 특히 **적군의 의도와 아군의 전술적 반응**을 데이터 변화를 근거로 심층 분석해야 합니다."

### 3. 데이터 구조 정의 (테이블 스키마 제시)
"제공되는 데이터는 **아군(KPA)과 적군(ROK) 간의 16 틱($\text{Tic}$) 교전 시나리오**의 일부 또는 전체이며, 다음 9개 열($\text{Column}$)로 구성되어 있습니다. 각 열의 의미는 다음과 같습니다.

1.  **시간:** $\text{Tic}$ 단위의 시간 흐름 (0부터 15까지)
2.  **식별:** 아군인지 적군인지 판단하는 기준 (숫자, 예: 0=아군, 1=적군)
3.  **플랫폼:** 플랫폼 이름
4.  **x좌표:** 현재 부대의 지도상 x좌표
5.  **y좌표:** 현재 부대의 지도상 y좌표
6.  **탐지거리:** 센서 등으로 아군을 탐지할 수 있는 최대 거리(숫자, 예: 0=탐지장비 없음, 100=탐지거리 100)
7.  **탐지여부:** 현재 플랫폼이 아군 또는 적군에게 탐지되었는지 여부(숫자, 0=아군이므로 탐지 불필요, 1=탐지되지 않은 적군, 2=탐지된 적군)
8.  **전튜력:** 현재 플랫폼이 가지고 있는 전투력(숫자, 0=파괴된 상태, 50=반파된 상태, 100=파괴된곳 없는 완전한 상태)
9.  **상황:** 현재 시간(tic) 별 대략적인 상황 설명

### 4. 분석 및 묘사 지침 (핵심 요구사항)
"아래의 분석 단계를 반드시 따르십시오.

1.  **데이터 변화 해석:** 각 섹션에서 **가장 극적인 변화(Drop/Spike)**를 보인 데이터(예: 탐지여부 1 $\rightarrow$ 2, 체력 100 $\rightarrow$ 75 등)를 추출하고, 이 변화가 **군사적으로 어떤 의미**를 가지는지 해석합니다.
2.  **적군 의도 분석:** **적군 위치, 방향, 속도**의 변화를 기반으로 적군 부대의 **전술적 의도**와 **북한군 교리($\text{Fix}$ 전술)** 중 어느 단계에 해당하는지 유추하여 설명합니다.
3.  **전반적인 상황 묘사:** 분석 내용을 종합하여 각 시간($\text{Tic}$)의 상황을 **교전 보고서** 형식의 문장으로 상세하게 묘사합니다. **KPA, ROK, C2, EW** 등의 전문 용어를 자연스럽게 사용합니다.

### 5. 데이터 입력 요청

"이제 다음 테이블 데이터를 분석하고, 위에 제시된 지침에 따라 상세한 군사 시나리오 묘사를 작성하십시오.

{table_data}"
"""

explain_template = ChatPromptTemplate.from_template(scenario_explain_template)

scenario_explain_chain = (
    RunnablePassthrough.assign(
        table_data = sql_query_gen_and_exec_chain
    )
    | explain_template
)

res = scenario_explain_chain.invoke({"question": question})


KeyError: "Input to ChatPromptTemplate is missing variables {'Column', 'Tic', 'Fix'}.  Expected: ['Column', 'Fix', 'Tic', 'table_data'] Received: ['question', 'table_data']\nNote: if you intended {Column} to be part of the string and not a variable, please escape it with double curly braces like: '{{Column}}'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT "